Imports and Setup

In [12]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Advanced ML libraries loaded successfully!")

Advanced ML libraries loaded successfully!


*Load the Datasets*

Ensure your files are in the same directory, or adjust the path if you are using Kaggle's

In [13]:
bbb_df = pd.read_csv('train_IPL.csv')
public_lb = pd.read_csv('public_lb_matches.csv')
schedule = pd.read_csv('schedule.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Training data raw shape: {bbb_df.shape}")

Training data raw shape: (272704, 38)


Entity Consolidation (Cleaning Names)

In [14]:
team_mapping = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

def clean_team_names(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].replace(team_mapping)
    return df

# Clean the raw datasets
bbb_df = clean_team_names(bbb_df, ['Bat First', 'Bat Second', 'toss_winner', 'match_won_by'])
public_lb = clean_team_names(public_lb, ['team_a', 'team_b', 'toss_winner'])
schedule = clean_team_names(schedule, ['team_a', 'team_b'])

Data Aggregation & Target Derivation

In [15]:
# Group ball-by-ball data to match level
match_df = bbb_df.groupby('Match ID').agg({
    'Bat First': 'first',
    'Bat Second': 'first',
    'Venue': 'first',
    'toss_winner': 'first',
    'toss_decision': 'first',
    'match_won_by': 'first'
}).reset_index()

# Calculate Innings Totals and Wickets
innings_stats = bbb_df.groupby(['Match ID', 'Innings']).agg({
    'Runs From Ball': 'sum',
    'Extra Runs': 'sum',
    'Wicket': 'sum'
}).reset_index()
innings_stats['Total Runs'] = innings_stats['Runs From Ball'] + innings_stats['Extra Runs']

targets = []
for _, row in match_df.iterrows():
    match_id = row['Match ID']
    team_a = row['Bat First']
    team_b = row['Bat Second']
    winner = row['match_won_by']
    
    match_innings = innings_stats[innings_stats['Match ID'] == match_id]
    try:
        runs_A = match_innings[match_innings['Innings'] == 1]['Total Runs'].values[0]
        wkts_B = match_innings[match_innings['Innings'] == 2]['Wicket'].values[0]
    except IndexError:
        targets.append(np.nan)
        continue

    if winner == team_a:
        margin_runs = runs_A - match_innings[match_innings['Innings'] == 2]['Total Runs'].values[0] if len(match_innings[match_innings['Innings'] == 2]) > 0 else 100
        targets.append('A_big' if margin_runs > 20 else 'A_small')
    elif winner == team_b:
        margin_wickets = 10 - wkts_B
        targets.append('B_big' if margin_wickets >= 6 else 'B_small')
    else:
        targets.append(np.nan)

match_df['target'] = targets
match_df = match_df.dropna(subset=['target'])

label_encoder = LabelEncoder()
match_df['target_encoded'] = label_encoder.fit_transform(match_df['target'])
print(f"Aggregated training matches: {match_df.shape[0]}")

Aggregated training matches: 1124


## 5. Historical Statistical Feature Engineering Engine

In [16]:
# Extract overall historical statistics per team to serve as powerful continuous features
all_teams = pd.concat([match_df['Bat First'], match_df['Bat Second']]).unique()
team_stats = {}

for team in all_teams:
    total_matches = match_df[(match_df['Bat First'] == team) | (match_df['Bat Second'] == team)].shape[0]
    wins = match_df[match_df['match_won_by'] == team].shape[0]
    bat_1st_matches = match_df[match_df['Bat First'] == team]
    bat_1st_wins = bat_1st_matches[bat_1st_matches['match_won_by'] == team].shape[0]
    
    team_stats[team] = {
        'win_rate': wins / total_matches if total_matches > 0 else 0.5,
        'bat_first_win_rate': bat_1st_wins / bat_1st_matches.shape[0] if bat_1st_matches.shape[0] > 0 else 0.5
    }

# Venue base scoring profile
venue_avg_runs = innings_stats.groupby('Match ID')['Total Runs'].sum().reset_index().merge(match_df[['Match ID', 'Venue']], on='Match ID')
venue_profile = venue_avg_runs.groupby('Venue')['Total Runs'].mean().to_dict()

def build_advanced_features(df, is_train=True):
    df_feats = df.copy()
    
    # Map historical profiles
    df_feats['team_a_win_rate'] = df_feats['Bat First'].map(lambda x: team_stats.get(x, {}).get('win_rate', 0.5))
    df_feats['team_b_win_rate'] = df_feats['Bat Second'].map(lambda x: team_stats.get(x, {}).get('win_rate', 0.5))
    df_feats['team_a_bat1st_rate'] = df_feats['Bat First'].map(lambda x: team_stats.get(x, {}).get('bat_first_win_rate', 0.5))
    df_feats['venue_avg_score'] = df_feats['Venue'].map(lambda x: venue_profile.get(x, 330.0))
    
    # Strengths differentials
    df_feats['win_rate_diff'] = df_feats['team_a_win_rate'] - df_feats['team_b_win_rate']
    
    # Categorical Label Encoding
    cat_cols = ['Bat First', 'Bat Second', 'Venue', 'toss_decision']
    for col in cat_cols:
        if col in df_feats.columns:
            df_feats[col + '_enc'] = df_feats[col].astype('category').cat.codes

    # Toss situational flag
    if 'toss_winner' in df_feats.columns:
        df_feats['team_a_won_toss'] = (df_feats['toss_winner'] == df_feats['Bat First']).astype(int)
    else:
        df_feats['team_a_won_toss'] = 0

    feature_cols = [
        'team_a_win_rate', 'team_b_win_rate', 'team_a_bat1st_rate', 'venue_avg_score',
        'win_rate_diff', 'Bat First_enc', 'Bat Second_enc', 'Venue_enc', 'toss_decision_enc', 'team_a_won_toss'
    ]
    
    if is_train:
        return df_feats[feature_cols], df_feats['target_encoded']
    return df_feats[feature_cols]

X_train, y_train = build_advanced_features(match_df, is_train=True)
print("Advanced momentum and history features engineered successfully!")

Advanced momentum and history features engineered successfully!


## 6. Optuna Hyperparameter Optimization with Built-In Calibration

In [17]:
def objective(trial):
    xgb_params = {
        'objective': 'multi:softprob',
        'num_class': 4,
        'eval_metric': 'mlogloss',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'subsample': trial.suggest_float('subsample', 0.65, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 0.95),
        'n_estimators': trial.suggest_int('n_estimators', 60, 200)
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    log_losses = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Wrap our model in Isotonic Calibration to avoid Log Loss confidence penalties
        base_model = xgb.XGBClassifier(**xgb_params, random_state=42)
        calibrated_clf = CalibratedClassifierCV(estimator=base_model, method='isotonic', cv=3)
        calibrated_clf.fit(X_tr, y_tr)
        
        preds = calibrated_clf.predict_proba(X_va)
        log_losses.append(log_loss(y_va, preds, labels=[0, 1, 2, 3]))
        
    return np.mean(log_losses)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print(f"Optimized Calibrated Log Loss achieved: {study.best_value:.4f}")

Optimized Calibrated Log Loss achieved: 1.3654


## 7. Train Final High-Precision Model Ensemble

In [18]:
# 1. Train Optimized Calibrated XGBoost Model
best_xgb_params = study.best_params.copy()
best_xgb_params.update({'objective': 'multi:softprob', 'num_class': 4})
base_xgb = xgb.XGBClassifier(**best_xgb_params, random_state=42)

final_xgb_model = CalibratedClassifierCV(estimator=base_xgb, method='isotonic', cv=5)
final_xgb_model.fit(X_train, y_train)

# 2. Train Robust Complementary Calibrated LightGBM Model for Ensembling
base_lgb = lgb.LGBMClassifier(objective='multiclass', num_class=4, learning_rate=0.03, max_depth=4, n_estimators=100, random_state=42, verbose=-1)
final_lgb_model = CalibratedClassifierCV(estimator=base_lgb, method='isotonic', cv=5)
final_lgb_model.fit(X_train, y_train)

print("Ensemble structures trained and calibrated flawlessly.")

Ensemble structures trained and calibrated flawlessly.


## 8. Process Test Set Inputs

In [19]:
def get_bat_first(row):
    if row['toss_decision'] == 'bat': return row['toss_winner']
    return row['team_b'] if row['toss_winner'] == row['team_a'] else row['team_a']

def get_bat_second(row):
    if row['toss_decision'] == 'field': return row['toss_winner']
    return row['team_b'] if row['toss_winner'] == row['team_a'] else row['team_a']

public_lb['actual_bat_first'] = public_lb.apply(get_bat_first, axis=1)
public_lb['actual_bat_second'] = public_lb.apply(get_bat_second, axis=1)

public_lb_test = pd.DataFrame({
    'Match ID': public_lb['match_id'],
    'Bat First': public_lb['actual_bat_first'],
    'Bat Second': public_lb['actual_bat_second'],
    'Venue': public_lb['venue'],
    'toss_decision': public_lb['toss_decision'],
    'toss_winner': public_lb['toss_winner']
})

private_lb_test = pd.DataFrame({
    'Match ID': schedule['match_id'],
    'Bat First': schedule['team_a'],
    'Bat Second': schedule['team_b'],
    'Venue': schedule['venue'],
    'toss_decision': 'field',
    'toss_winner': schedule['team_b']
})

combined_test = pd.concat([public_lb_test, private_lb_test], ignore_index=True)
X_test = build_advanced_features(combined_test, is_train=False)

## 9. Format, Normalize, and Export Final Predictions

In [23]:
# 1. Select the probability columns
prob_cols = ['A_small', 'A_big', 'B_small', 'B_big']

# 2. Round the first 3 columns to exactly 2 decimal places
submission['A_small'] = submission['A_small'].round(2)
submission['A_big'] = submission['A_big'].round(2)
submission['B_small'] = submission['B_small'].round(2)

# 3. Forcibly calculate the last column as the exact remainder so they sum perfectly to 1.00
submission['B_big'] = (1.0 - submission[['A_small', 'A_big', 'B_small']].sum(axis=1)).round(2)

# 4. Save the high-precision formatted file with strict Linux line endings
submission.to_csv('submission_fixed.csv', index=False, float_format='%.2f', lineterminator='\n')

print("Pipeline complete! 'submission_fixed.csv' has exactly 2 decimals and sums perfectly to 1.00.")
submission.head()

Pipeline complete! 'submission_fixed.csv' has exactly 2 decimals and sums perfectly to 1.00.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.18,0.35,0.18,0.29
1,1473489,0.22,0.24,0.20,0.34
2,1473490,0.23,0.24,0.18,0.35
3,1473491,0.21,0.25,0.19,0.35
4,1473492,0.19,0.22,0.23,0.36
